# EnergAIzer — CPU Colab Demo

Fast, analytical **GPU power / energy / latency** prediction for AI workloads — no GPU required (runs on a free Colab **CPU** runtime).

Source: https://github.com/shubhamOjha1000/single_kernel_GPU_model (mirror of the ISPASS'26 EnergAIzer artifact).

Four use cases:
0. **Predict a single kernel** (one GEMM)
1. **Predict energy of an AI model** (BERT / ResNet / ViT …)
2. **Voltage–frequency scaling** (DVFS sweep on A100)
3. **Design-space exploration** (extrapolate A100 → H100)

> Run the 3 setup cells first (clone + install + download the LUT database), then any section. The Google Drive download is the one step that can occasionally fail due to quotas — see the note in that cell.

## Setup — Step 1: clone the repo + install CPU dependencies

In [ ]:
import os, sys, subprocess

REPO_URL = "https://github.com/shubhamOjha1000/single_kernel_GPU_model.git"
ROOT = "/content/single_kernel_GPU_model"
PKG  = os.path.join(ROOT, "energaizer-ispass26-artifact-main")

# 1) Clone the repo (the Python package `gee` lives inside PKG)
if not os.path.isdir(PKG):
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, ROOT], check=True)
print("Repo at:", PKG)

# 2) Install CPU-only deps. Inference from queries / workload JSONs does NOT need
#    torch or transformers (those are only for *generating* new workload JSONs).
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "numpy", "pandas", "scipy", "cvxpy", "pyyaml",
                "opt-einsum", "tqdm", "scikit-learn", "gdown"], check=True)
print("Dependencies installed.")

## Setup — Step 2: download the pre-collected LUT database

EnergAIzer needs an offline database of measured kernel latency/power (the lookup tables the analytical model is fitted against). It is **not** in the git repo and is downloaded from Google Drive (~tens of MB).

If `gdown` hits a Drive quota error, re-run this cell later, or download the file manually from the printed link and upload it to `PKG/database/data/`.

In [ ]:
import os, glob, subprocess

DB_BASE = os.path.join(PKG, "database", "data")
os.makedirs(DB_BASE, exist_ok=True)
DB_FILE_ID = "1krvqRFDnaqrJUT06V2psIua0wQr6ETAE"

def find_luts():
    return glob.glob(os.path.join(DB_BASE, "**", "*.csv"), recursive=True)

if not find_luts():
    archive = os.path.join(DB_BASE, "precollected_database.tar.gz")
    try:
        import gdown
        gdown.download(id=DB_FILE_ID, output=archive, quiet=False)
    except Exception as e:
        print("gdown failed:", e)
    if os.path.exists(archive):
        subprocess.run(["tar", "-xzf", archive, "-C", DB_BASE], check=True)
        os.remove(archive)
    else:
        # Fallback: the repo's curl-based downloader
        subprocess.run(["bash", os.path.join(PKG, "misc", "gdrive_download.sh"),
                        "https://drive.google.com/file/d/%s/view" % DB_FILE_ID],
                       cwd=DB_BASE, check=True)

luts = find_luts()
print("Found %d LUT csv files." % len(luts))
print("Manual link (if needed): https://drive.google.com/file/d/%s/view" % DB_FILE_ID)
assert luts, "LUT database not found - download failed. See the note above."

## Setup — Step 3: normalize LUT layout

The LUT config references CSVs by bare filename, resolved against `lut_folder_abs_path`. Ensure every CSV sits directly in `database/data/` (symlink up if the archive extracted into a subfolder).

In [ ]:
import os, glob

for csv in glob.glob(os.path.join(DB_BASE, "**", "*.csv"), recursive=True):
    dest = os.path.join(DB_BASE, os.path.basename(csv))
    if os.path.abspath(csv) != os.path.abspath(dest) and not os.path.exists(dest):
        os.symlink(os.path.abspath(csv), dest)

flat = sorted(f for f in os.listdir(DB_BASE) if f.endswith(".csv"))
print("%d CSVs directly in database/data:" % len(flat))
for f in flat:
    print("  ", f)

## 0. Predict a single kernel

Build the estimator once (loads all lookup tables), then query a single **bfloat16 Tensor-Core GEMM** on an A100-40GB-PCIE (`yz8`) at 900 MHz. Returns `(latency, power, energy)`.

In [ ]:
import sys, os
if PKG not in sys.path:
    sys.path.insert(0, PKG)

from gee import get_gee

A100_GPU   = os.path.join(PKG, "config", "gpu", "yz8.yaml")          # yz8 == A100-40GB-PCIE
LUT_CONFIG = os.path.join(PKG, "experiments_endtoend", "exp_config", "a100_lut_config.yaml")
VOLT_JSON  = os.path.join(PKG, "config", "dvfs", "yz8", "supply_voltage.json")
IDLE_JSON  = os.path.join(PKG, "config", "dvfs", "yz8", "idle_power.json")

estimator = get_gee(
    gpu_yaml_path=A100_GPU,
    lut_yaml_path=LUT_CONFIG,
    dvfs_aware=True,
    dvfs_inference_mode="all",
    dvfs_supply_voltage_json=VOLT_JSON,
    dvfs_idle_power_json=IDLE_JSON,
    lut_folder_abs_path=DB_BASE,
)
print("Estimator built.")

In [ ]:
# A single bf16 Tensor-Core GEMM:  (batch x M x K) @ (batch x K x N)
query = {
    "batch": 1,
    "dimM": 4096,
    "dimN": 4096,
    "dimK": 4096,
    "precM": "bf16",        # operand precision
    "precA": "bf16",        # accumulate / output precision
    "useTensorCore": True,
}
query_type = ("gemm",
              "tc" if query["useTensorCore"] else "cuda",
              "{}_{}".format(query["precM"], query["precA"]))

latency, power, energy = estimator.lookup(query, query_type,
                                          target_freq=900, lookup_target="all")

print("GEMM  batch=1, M=N=K=4096  (bf16, Tensor Core)  @ A100, 900 MHz")
print("  latency :", latency)
print("  power   :", power, "W")
print("  energy  :", energy)

## 1. Predict energy consumption of an AI model

Run the end-to-end estimator over a few provided workload JSONs (BERT, ResNet101, ViT). Each JSON is a decomposed list of kernels for a model+shape. Prior-work baselines are disabled (`--no_neusight/--no_limicro/--no_roofline`) since they need extra downloads. Output is a CSV with per-workload latency and energy.

> A small subset is used to keep it fast — point `--workload_folder` at `test/data/workloads/all` for the full set.

In [ ]:
import os, shutil, subprocess
import pandas as pd

ENDTOEND = os.path.join(PKG, "experiments_endtoend")

SRC = os.path.join(PKG, "test", "data", "workloads", "all")
SUB = os.path.join(PKG, "test", "data", "workloads", "demo_subset")
os.makedirs(SUB, exist_ok=True)
for p in ["bertmodel_large_pbf16_b8_s128_modeprefill.json",
          "resnet101__pbf16_b8_freq900.json",
          "vitmodel_large_pbf16_b16_freq900.json"]:
    src = os.path.join(SRC, p)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(SUB, p))
print("Subset workloads:", os.listdir(SUB))

cmd = [sys.executable, "run_estimators.py",
       "--workload_folder", SUB,
       "--result_save_to", "results/a100_endtoend",
       "--result_filename", "demo.csv",
       "--gpu_config_yaml", "../config/gpu/yz8.yaml",
       "--lut_config_yaml", "exp_config/a100_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--target_freq", "900",
       "--flash_attention_enable",
       "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

pd.set_option("display.max_columns", None)
df = pd.read_csv(os.path.join(ENDTOEND, "results/a100_endtoend/demo.csv"))
df

## 2. Voltage–frequency scaling (DVFS)

Same estimator, but `--dvfs_aware` sweeps the core frequency 510 → 1410 MHz (using the A100 supply-voltage and idle-power profiles). The output shows how predicted latency/power/energy change with frequency for each workload.

In [ ]:
import os, shutil, subprocess
import pandas as pd

DVFS_SRC = os.path.join(PKG, "test", "data", "workloads", "dvfs")
DVFS_SUB = os.path.join(PKG, "test", "data", "workloads", "dvfs_demo")
os.makedirs(DVFS_SUB, exist_ok=True)
for p in ["gpt2model_gpt2_pbf16_b8_s128_modeprefill.json",
          "optmodel_opt-1p3b_pbf16_b2_s128_modeprefill.json"]:
    src = os.path.join(DVFS_SRC, p)
    if os.path.exists(src):
        shutil.copy(src, os.path.join(DVFS_SUB, p))
print("DVFS workloads:", os.listdir(DVFS_SUB))

cmd = [sys.executable, "run_estimators.py",
       "--workload_folder", DVFS_SUB,
       "--result_save_to", "results/a100_dvfs",
       "--result_filename", "dvfs_demo.csv",
       "--gpu_config_yaml", "../config/gpu/yz8.yaml",
       "--lut_config_yaml", "exp_config/a100_dvfs_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--dvfs_aware",
       "--dvfs_supply_voltage_json", "../config/dvfs/yz8/supply_voltage.json",
       "--dvfs_idle_power_json", "../config/dvfs/yz8/idle_power.json",
       "--freq_min", "510", "--freq_max", "1410", "--freq_step", "180",
       "--flash_attention_enable",
       "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

df = pd.read_csv(os.path.join(ENDTOEND, "results/a100_dvfs/dvfs_demo.csv"))
df

## 3. Design-space exploration (A100 → H100 extrapolation)

EnergAIzer can forecast a *different* GPU architecture by re-using the A100-fitted hardware constants and scaling by the target GPU's SM count / bandwidth / Tensor-Core throughput. Here we extrapolate to an **H100-SXM @ 1830 MHz, 1.0 V** with no H100 measurements.

In [ ]:
import os, subprocess
import pandas as pd

cmd = [sys.executable, "run_estimators.py",
       "--workload_folder", SUB,                       # reuse the subset from section 1
       "--result_save_to", "results/h100_extrap",
       "--result_filename", "h100_demo.csv",
       "--multiple_config",
       "--multiple_gpu_configs_yaml", "../config/multiple/gpu.yaml",
       "--multiple_gpu_supply_voltage_yaml", "../config/multiple/supply_voltage.yaml",
       "--multiple_gpu_idle_power_yaml", "../config/multiple/idle_power.yaml",
       "--lut_config_yaml", "exp_config/extrapolation_lut_config.yaml",
       "--lut_folder_abs_path", DB_BASE,
       "--no_neusight", "--no_limicro", "--no_roofline",
       "--extrapolation",
       "--target_gpu_config_yaml", "../config/gpu/h100sxm.yaml",
       "--target_supply_voltage_value", "1000",
       "--target_freq", "1830",
       "--target_supply_voltage_json", "../config/dvfs/h100/supply_voltage.json",
       "--target_idle_power_json", "../config/dvfs/h100/idle_power.json",
       "--flash_attention_enable",
       "--flash_attention_estimate_method", "flashattention_v2"]
subprocess.run(cmd, cwd=ENDTOEND, check=True)

df = pd.read_csv(os.path.join(ENDTOEND, "results/h100_extrap/h100_demo.csv"))
df

## Recap

- **Section 0** used the `gee` Python API directly for one kernel.
- **Sections 1–3** drove `experiments_endtoend/run_estimators.py` over workload JSONs for whole-model energy, DVFS sweeps, and cross-architecture extrapolation.

All of it ran on CPU — no GPU and no profiling. To predict your own model, generate workload JSONs per `test/code/README.md`, drop them in a folder, and point `--workload_folder` at it.